# MEP Tutorial Part 1: Building Topology and Graph Representation

This notebook is the first part of a tutorial on using topologic_fast for MEP
(Mechanical, Electrical, Plumbing) system analysis.

**Adapted from topologicpy MEP01 notebook**

This program is free software under the GNU Affero General Public License.

## What You'll Learn

1. Creating building geometry with CellComplex
2. Understanding room connectivity
3. Creating a dual graph from building topology
4. Visualizing the graph with directional edges

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Part 1: Create Building Geometry

We'll create a simple building with:
- Main building volume (3 units wide corridor)
- A corridor running through
- A source room (mechanical room)

The geometry represents a simple floor plate with rooms along a corridor.

In [ ]:
# Building parameters
floor_height = 3.0  # meters

# Create the main building - a row of rooms
rooms = []
room_names = []
room_types = []

# Left wing rooms (3 rooms)
for i in range(3):
    room = tf.Cell.Box(0, i * 4, 0, 4, 4, floor_height)
    rooms.append(room)
    room_names.append(f"Room L{i+1}")
    room_types.append("room")

# Right wing rooms (3 rooms)
for i in range(3):
    room = tf.Cell.Box(6, i * 4, 0, 4, 4, floor_height)
    rooms.append(room)
    room_names.append(f"Room R{i+1}")
    room_types.append("room")

# Corridor (center)
corridor = tf.Cell.Box(4, 0, 0, 2, 12, floor_height)
rooms.append(corridor)
room_names.append("Corridor")
room_types.append("corridor")

# Source room (mechanical/HVAC)
source = tf.Cell.Box(4, -2, 0, 2, 2, floor_height)
rooms.append(source)
room_names.append("Mech Room")
room_types.append("source")

print(f"Created {len(rooms)} rooms:")
for name, rtype in zip(room_names, room_types):
    print(f"  - {name} ({rtype})")

## Part 2: Create CellComplex

Combine all rooms into a CellComplex. This represents the building as a
topologically connected structure where we can query adjacency relationships.

In [ ]:
# Create the building as a CellComplex
building = tf.CellComplex.ByCells(rooms)

print(f"Building CellComplex:")
print(f"  Number of cells (rooms): {building.NumCells()}")
print(f"  Total volume: {building.Volume():.1f} m^3")
print(f"  Total surface area: {building.Area():.1f} m^2")

## Part 3: Create Dual Graph

The dual graph represents:
- **Vertices**: Room centroids
- **Edges**: Connections between adjacent rooms (shared walls)

This graph is essential for MEP routing analysis.

In [ ]:
# Create graph from building topology
graph = tf.Graph.ByTopology(building)

print(f"Dual Graph:")
print(f"  Vertices (rooms): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()} steps")

## Part 4: Analyze Connectivity

Examine which rooms are connected to which. This is crucial for
understanding how MEP systems need to be routed.

In [ ]:
graph_vertices = graph.Vertices()

# Match vertices to rooms by centroid position
def get_room_index(vertex):
    """Find the room index for a graph vertex based on centroid matching."""
    coords = vertex.Coordinates()
    for i, room in enumerate(rooms):
        cx, cy, cz = room.CenterOfMass()
        if abs(coords[0] - cx) < 0.1 and abs(coords[1] - cy) < 0.1:
            return i
    return -1

print("Room Connectivity:")
print("=" * 50)

for i, v in enumerate(graph_vertices):
    room_idx = get_room_index(v)
    if room_idx >= 0:
        name = room_names[room_idx]
        rtype = room_types[room_idx]
    else:
        coords = v.Coordinates()
        name = f"Unknown ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})"
        rtype = "unknown"
    
    degree = graph.VertexDegree(v)
    adjacent = graph.AdjacentVertices(v)
    
    print(f"\n{name} [{rtype}]: {degree} connections")
    for adj_v in adjacent:
        adj_idx = get_room_index(adj_v)
        if adj_idx >= 0:
            adj_name = room_names[adj_idx]
        else:
            adj_coords = adj_v.Coordinates()
            adj_name = f"Unknown ({adj_coords[0]:.1f}, {adj_coords[1]:.1f})"
        print(f"    -> {adj_name}")

## Part 5: Visualization

Create a 3D visualization of the building and its connectivity graph.

In [ ]:
def visualize_building_and_graph(building, graph, room_names, room_types):
    """Create a 3D visualization of the building with its dual graph."""
    fig = go.Figure()
    
    # Color scheme
    color_map = {
        'room': '#90EE90',      # Light green
        'corridor': '#D3D3D3',   # Light gray
        'source': '#FFD700'      # Gold
    }
    
    # Plot building cells
    cells = building.Cells()
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        rtype = room_types[i] if i < len(room_types) else 'room'
        color = color_map.get(rtype, '#90EE90')
        
        for j, face in enumerate(faces):
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.4,
                    alphahull=0,
                    name=room_names[i] if i < len(room_names) else f'Cell {i}',
                    showlegend=(j == 0)
                ))
    
    # Plot graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='red', width=5),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plot graph vertices
    graph_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=8, color='blue'),
        name='Room Centroids',
        hovertext=[f'Room {i}' for i in range(len(vertex_coords))],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title='Building with Connectivity Graph',
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    return fig

In [ ]:
fig = visualize_building_and_graph(building, graph, room_names, room_types)
fig.show()

## Part 6: 2D Floor Plan View

In [ ]:
def visualize_2d_floorplan(building, graph, room_names, room_types):
    """Create a 2D top-down view of the floor plan."""
    fig = go.Figure()
    
    color_map = {
        'room': '#90EE90',
        'corridor': '#D3D3D3',
        'source': '#FFD700'
    }
    
    cells = building.Cells()
    
    # Plot rooms as 2D shapes
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        rtype = room_types[i] if i < len(room_types) else 'room'
        color = color_map.get(rtype, '#90EE90')
        name = room_names[i] if i < len(room_names) else f'Cell {i}'
        
        # Find bottom face (z = 0)
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            z_coords = [c[2] for c in coords]
            
            if all(abs(z) < 0.01 for z in z_coords):
                x = [c[0] for c in coords] + [coords[0][0]]
                y = [c[1] for c in coords] + [coords[0][1]]
                
                fig.add_trace(go.Scatter(
                    x=x, y=y,
                    fill='toself',
                    fillcolor=color,
                    line=dict(color='black', width=2),
                    name=name,
                    hoverinfo='name'
                ))
                break
    
    # Plot graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='rgba(255,0,0,0.6)', width=4),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plot graph vertices
    graph_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(size=12, color='red', line=dict(color='darkred', width=2)),
        name='Graph Nodes',
        hoverinfo='text',
        hovertext=[f'Node {i}' for i in range(len(vertex_coords))]
    ))
    
    fig.update_layout(
        title='2D Floor Plan with Connectivity Graph',
        xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y (m)'),
        width=800,
        height=700,
        showlegend=True
    )
    
    return fig

fig_2d = visualize_2d_floorplan(building, graph, room_names, room_types)
fig_2d.show()

## Part 7: Understanding Tree Graphs for MEP

In MEP systems, we often need a tree graph rooted at the source (mechanical room).
This represents the distribution hierarchy.

**Note:** `Graph.Tree()` is not yet implemented in topologic_fast.
The concept is shown below with manual path finding.

In [ ]:
# Find the source room vertex
source_idx = room_types.index('source')
source_room = rooms[source_idx]
source_centroid = source_room.CenterOfMass()

# Find the corresponding graph vertex
source_vertex = graph.NearestVertex(tf.Vertex.ByCoordinates(*source_centroid))

print(f"Source room (Mechanical Room):")
print(f"  Centroid: ({source_centroid[0]:.2f}, {source_centroid[1]:.2f}, {source_centroid[2]:.2f})")

# Calculate distances from source to all rooms
print(f"\nDistances from Source to All Rooms:")
print("-" * 40)

distances = []
for i, v in enumerate(graph_vertices):
    room_idx = get_room_index(v)
    if room_idx >= 0:
        name = room_names[room_idx]
    else:
        name = f"Node {i}"
    
    dist = graph.Distance(source_vertex, v)
    distances.append((name, dist))

# Sort by distance
distances.sort(key=lambda x: x[1])

for name, dist in distances:
    print(f"  {name}: {dist} steps")

In [ ]:
# Create depth map from source
depth_map = graph.DepthMap(source_vertex)

print("Depth Map (BFS levels from source):")
print("=" * 40)
for i, depth in enumerate(depth_map):
    room_idx = get_room_index(graph_vertices[i])
    if room_idx >= 0:
        name = room_names[room_idx]
    else:
        name = f"Node {i}"
    print(f"  Level {depth}: {name}")

## Summary

In this tutorial, we covered:

1. **Building Geometry**: Creating rooms as Cell objects and combining them into a CellComplex
2. **Dual Graph**: Using `Graph.ByTopology()` to create a connectivity graph
3. **Connectivity Analysis**: Examining adjacency relationships between rooms
4. **Visualization**: Both 3D and 2D views of the building and graph
5. **Distance Analysis**: Computing graph distances for MEP routing

### Not Yet Implemented in topologic_fast:
- `Graph.Tree()` - Creating spanning tree from a root vertex
- `Topology.AddApertures()` - Adding openings (doors, windows)
- `Dictionary` class - For storing attributes on topologies
- `CellComplex.Decompose()` - Decomposing into internal/external faces

### Next Steps (MEP02)
- Creating MEP network with duct dimensions
- Calculating pressure drops
- Optimizing routing paths